In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
test_df = pd.read_csv('/content/test_split.csv')
train_df = pd.read_csv('/content/train_split.csv')
val_df = pd.read_csv('/content/val_split.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
train_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,155515,65240,126963,Which are the best books for iit jam mathematics?,What are the best books for IIT-JAM mathematics?,1
1,400580,533929,1378,What are the best Photo startups?,What is a good photo?,0
2,101326,49803,45910,How can I find a job?,What is the fastest way to find a job?,1
3,87072,146702,146703,Which water purifier company provide best afte...,"I am planning to purchase water purifier, whic...",1
4,67927,25123,117478,Can I add an app to my Vizio smart TV?,Are there any YouTube like apps for sharing on...,0


In [5]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generate Embeddings for your test split

In [6]:
question1 = test_df['question1'].to_list()
question2 = test_df['question2'].to_list()

embedding1 = model.encode(question1, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
embedding2 = model.encode(question2, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/948 [00:00<?, ?it/s]

Batches:   0%|          | 0/948 [00:00<?, ?it/s]

In [7]:
import numpy as np

def cosine_sim_batch(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.sum(a_norm * b_norm, axis=1)

similarities = cosine_sim_batch(embedding1, embedding2)
test_df['similarity'] = similarities

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold = 0.75
predictions = (test_df['similarity'] >= threshold).astype(int)
labels = test_df['is_duplicate']

print("Accuracy:", accuracy_score(labels, predictions))
print("Precision:", precision_score(labels, predictions))
print("Recall:", recall_score(labels, predictions))
print("F1:", f1_score(labels, predictions))

Accuracy: 0.7669019193984565
Precision: 0.6342637777344005
Recall: 0.87074586869138
F1: 0.7339256136124078


In [9]:
test_df.head()

,id,qid1,qid2,question1,question2,is_duplicate,similarity
0,245868,358798,148076,Are snails born with their shells?,Are snails born with shells or do they grow?,1,0.955840
1,11393,22003,22004,How do I get more topic diversity in my Quora ...,How do I get more topics?,1,0.603999
2,371228,501843,501844,Why is it important to have hobbies?,Why should we have hobbies?,1,0.907511
3,80111,136259,136260,What are the largest cities in CK2?,What are the largest cities in North India?,0,0.641329
4,140957,223900,53245,How do I calculate the torque required on bicy...,How do I calculate the required torque of a ve...,0,0.846118


Checking Different values of threshold

In [10]:


thresholds = np.arange(0.3, 0.95, 0.05)

print(f"{'Threshold':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
for t in thresholds:
    preds = (test_df['similarity'] >= t).astype(int)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    print(f"{t:<10.2f} {acc:<10.4f} {prec:<10.4f} {rec:<10.4f} {f1:<10.4f}")

Threshold  Accuracy   Precision  Recall     F1        
0.30       0.4815     0.4159     1.0000     0.5875    
0.35       0.5025     0.4260     1.0000     0.5974    
0.40       0.5271     0.4384     0.9998     0.6096    
0.45       0.5578     0.4550     0.9995     0.6253    
0.50       0.5916     0.4748     0.9983     0.6435    
0.55       0.6314     0.5004     0.9956     0.6660    
0.60       0.6713     0.5293     0.9888     0.6895    
0.65       0.7094     0.5615     0.9720     0.7118    
0.70       0.7434     0.5971     0.9381     0.7297    
0.75       0.7669     0.6343     0.8707     0.7339    
0.80       0.7799     0.6788     0.7669     0.7202    
0.85       0.7721     0.7255     0.6156     0.6660    
0.90       0.7453     0.7850     0.4269     0.5531    


Threshold: 0.75 | Accuracy: 0.767 | Precision: 0.634 | Recall: 0.871 | F1: 0.734 -  finalize these values

In [13]:
np.save('/content/q1_embeddings_baseline.npy', embedding1)
np.save('/content/q2_embeddings_baseline.npy', embedding2)
test_df.to_csv('/content/test_split_with_similarity.csv', index=False)